# EE 446 TinyML — Lab 3  
## Quantization of a DNN Using the UCI Human Activity Recognition Dataset



## 1. Environment Setup


In [ ]:
import sys
print(sys.executable)
%pip install "tensorflow==2.15.1" "tensorflow-model-optimization==0.8.0" "scikit-learn==1.4.2" "pandas==2.2.2" "matplotlib==3.8.4"

## 2. Imports and Reproducibility


In [ ]:
import os
import zipfile
import pathlib
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_model_optimization as tfmot

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)


## 3. Download and Extract the UCI HAR Dataset

The original dataset contains:
- **561 numerical features** extracted from smartphone sensor signals
- **6 activity classes**
- predefined **training** and **test** splits

In [ ]:
dataset_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip"
zip_path = "uci_har_dataset.zip"
extract_dir = "."

if not os.path.exists("UCI HAR Dataset"):
    !wget -q "{dataset_url}" -O "{zip_path}"
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    print("Dataset downloaded and extracted.")
else:
    print("Dataset directory already exists.")


## 4. Load the Data


In [ ]:
def load_har_data(root_dir="UCI HAR Dataset"):
    root_dir = pathlib.Path(root_dir)

    X_train = np.loadtxt(root_dir / "train" / "X_train.txt").astype(np.float32)
    y_train = np.loadtxt(root_dir / "train" / "y_train.txt").astype(np.int32) - 1
    X_test = np.loadtxt(root_dir / "test" / "X_test.txt").astype(np.float32)
    y_test = np.loadtxt(root_dir / "test" / "y_test.txt").astype(np.int32) - 1

    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = load_har_data()

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

# TODO: define num_features and num_classes
num_features = X_train.shape[1]
num_classes = len(class_names)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape :", X_test.shape)
print("y_test shape :", y_test.shape)
print("Number of features:", num_features)
print("Number of classes :", num_classes)

## 5. Quick Inspection


In [ ]:
label_counts = pd.Series(y_train).value_counts().sort_index()

dataset_summary = pd.DataFrame({
    "Class Index": list(range(num_classes)),
    "Class Name": class_names,
    "Training Samples": label_counts.values,
})

dataset_summary

## 6. Train a Baseline DNN


### Architecture
- Input: 561 features
- Dense(256, ReLU)
- Dense(128, ReLU)
- Dense(64, ReLU)
- Dense(6, Softmax)


In [ ]:
def build_baseline_model(input_dim, num_classes):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

baseline_model = build_baseline_model(num_features, num_classes)
baseline_model.summary()

### Train the Baseline Model


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True
    )
]

history = baseline_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=40,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

### Training Curves


In [ ]:
history_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df["accuracy"], label="Train Accuracy")
axes[0].plot(history_df["val_accuracy"], label="Validation Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Training vs Validation Accuracy")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history_df["loss"], label="Train Loss")
axes[1].plot(history_df["val_loss"], label="Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Training vs Validation Loss")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 7. Evaluate the Baseline Keras Model


In [ ]:
baseline_probs = baseline_model.predict(X_test, verbose=0)
baseline_preds = np.argmax(baseline_probs, axis=1)
baseline_test_acc = accuracy_score(y_test, baseline_preds)

print(f"Test Accuracy: {baseline_test_acc:.4f}\n")
print(classification_report(y_test, baseline_preds, target_names=class_names, digits=4))

disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, baseline_preds),
    display_labels=class_names
)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("Baseline Model - Confusion Matrix")
plt.show()

## 8. TensorFlow Lite Utilities

In [ ]:
def save_binary_model(model_content, filename):
    with open(filename, "wb") as f:
        f.write(model_content)
    return os.path.getsize(filename) / 1024.0  # KB

def representative_dataset_gen():
    for i in range(300):
        yield [X_train[i:i+1].astype(np.float32)]

def evaluate_tflite_model(tflite_model, X, y_true):
    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    input_scale, input_zero_point = input_details["quantization"]
    output_scale, output_zero_point = output_details["quantization"]

    y_pred = []

    for i in range(len(X)):
        x = X[i:i+1].astype(np.float32)

        # Quantize the input only when the input dtype is int8 or uint8.
        # Otherwise keep the input in the required floating-point dtype.
        if input_details["dtype"] == np.int8:
            x = np.round(x / input_scale + input_zero_point).astype(np.int8)
        elif input_details["dtype"] == np.uint8:
            x = np.round(x / input_scale + input_zero_point).astype(np.uint8)
        else:
            x = x.astype(input_details["dtype"])

        interpreter.set_tensor(input_details["index"], x)
        interpreter.invoke()

        output = interpreter.get_tensor(output_details["index"])

        # If the output is quantized, dequantize it back to float32.
        if output_details["dtype"] in (np.int8, np.uint8):
            output = (output.astype(np.float32) - output_zero_point) * output_scale

        y_pred.append(np.argmax(output, axis=1)[0])

    y_pred = np.array(y_pred)
    acc = accuracy_score(y_true, y_pred)
    return acc, y_pred

def convert_to_tflite_fp32(model):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    return converter.convert()

def convert_to_tflite_dynamic_range(model):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    return converter.convert()

def convert_to_tflite_float16(model):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    return converter.convert()

def convert_to_tflite_int8(model):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    return converter.convert()

## 9. Post-Training Quantization (PTQ)


In [ ]:
fp32_tflite_model = convert_to_tflite_fp32(baseline_model)
drq_tflite_model = convert_to_tflite_dynamic_range(baseline_model)
float16_tflite_model = convert_to_tflite_float16(baseline_model)
int8_tflite_model = convert_to_tflite_int8(baseline_model)

fp32_size_kb = save_binary_model(fp32_tflite_model, "baseline_fp32.tflite")
drq_size_kb = save_binary_model(drq_tflite_model, "baseline_dynamic_range.tflite")
float16_size_kb = save_binary_model(float16_tflite_model, "baseline_float16.tflite")
int8_size_kb = save_binary_model(int8_tflite_model, "baseline_int8.tflite")

fp32_acc, fp32_preds = evaluate_tflite_model(fp32_tflite_model, X_test, y_test)
drq_acc, drq_preds = evaluate_tflite_model(drq_tflite_model, X_test, y_test)
float16_acc, float16_preds = evaluate_tflite_model(float16_tflite_model, X_test, y_test)
int8_acc, int8_preds = evaluate_tflite_model(int8_tflite_model, X_test, y_test)

print(f"FP32 TFLite       - Accuracy: {fp32_acc:.4f}, Size: {fp32_size_kb:.2f} KB")
print(f"Dynamic Range PTQ - Accuracy: {drq_acc:.4f}, Size: {drq_size_kb:.2f} KB")
print(f"Float16 PTQ       - Accuracy: {float16_acc:.4f}, Size: {float16_size_kb:.2f} KB")
print(f"Int8 PTQ          - Accuracy: {int8_acc:.4f}, Size: {int8_size_kb:.2f} KB")

## 10. PTQ Comparison: Accuracy and Model Size


In [ ]:
ptq_results = pd.DataFrame([
    ["Baseline", "FP32 TFLite", fp32_acc, fp32_size_kb],
    ["Baseline", "Dynamic Range PTQ", drq_acc, drq_size_kb],
    ["Baseline", "Float16 PTQ", float16_acc, float16_size_kb],
    ["Baseline", "Int8 PTQ", int8_acc, int8_size_kb],
], columns=["Model Family", "Format", "Test Accuracy", "Model Size (KB)"])

ptq_results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(ptq_results["Format"], ptq_results["Model Size (KB)"], color="steelblue")
axes[0].set_ylabel("Model Size (KB)")
axes[0].set_title("PTQ Model Size Comparison")
axes[0].tick_params(axis="x", rotation=30)
axes[0].grid(axis="y")

axes[1].bar(ptq_results["Format"], ptq_results["Test Accuracy"], color="darkorange")
axes[1].set_ylabel("Test Accuracy")
axes[1].set_title("PTQ Test Accuracy Comparison")
axes[1].tick_params(axis="x", rotation=30)
axes[1].grid(axis="y")

plt.tight_layout()
plt.show()

### Confusion Matrix for the PTQ Int8 Model


In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, int8_preds),
    display_labels=class_names
)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("PTQ Int8 Model - Confusion Matrix")
plt.show()

print(classification_report(y_test, int8_preds, target_names=class_names, digits=4))

## 11. Quantization-Aware Training (QAT)

In [ ]:
qat_model = tfmot.quantization.keras.quantize_model(baseline_model)

qat_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

qat_model.summary()

### Fine-Tune the QAT Model


In [ ]:
qat_history = qat_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=8,
    batch_size=64,
    verbose=1
)

### Evaluate the QAT Keras Model


In [ ]:
qat_probs = qat_model.predict(X_test, verbose=0)
qat_preds_keras = np.argmax(qat_probs, axis=1)
qat_test_acc = accuracy_score(y_test, qat_preds_keras)

print(f"QAT Keras Test Accuracy: {qat_test_acc:.4f}\n")
print(classification_report(y_test, qat_preds_keras, target_names=class_names, digits=4))

### Convert the QAT Model to Int8 TensorFlow Lite


In [ ]:
qat_int8_tflite_model = convert_to_tflite_int8(qat_model)
qat_int8_size_kb = save_binary_model(qat_int8_tflite_model, "qat_int8.tflite")
qat_int8_acc, qat_int8_preds = evaluate_tflite_model(qat_int8_tflite_model, X_test, y_test)

print(f"QAT Int8 TFLite - Accuracy: {qat_int8_acc:.4f}, Size: {qat_int8_size_kb:.2f} KB")

## 12. PTQ Int8 vs QAT Int8


In [ ]:
ptq_vs_qat_results = pd.DataFrame([
    ["PTQ Int8", int8_acc, int8_size_kb],
    ["QAT Int8", qat_int8_acc, qat_int8_size_kb],
], columns=["Model", "Test Accuracy", "Model Size (KB)"])

ptq_vs_qat_results

In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, qat_int8_preds),
    display_labels=class_names
)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("QAT Int8 Model - Confusion Matrix")
plt.show()

print(classification_report(y_test, qat_int8_preds, target_names=class_names, digits=4))

### Answers

**1. Which quantization method gave the smallest model size?**
Int8 PTQ are expected to be the smallest, roughly a 4x reduction from the FP32 baseline, since both weights and activations are packed as 8-bit integers instead of 32-bit floats. Dynamic-range PTQ is close behind, and float16 typically lands at about half the FP32 size.

**2. Which quantization method gave the best accuracy among the TensorFlow Lite models?**
FP32 TFLite should match the original Keras baseline almost exactly, since no precision is lost in conversion. Among the compressed formats, float16 PTQ is usually closest to full precision, followed by dynamic-range PTQ, with int8 PTQ typically showing the largest, though usually still small, accuracy drop, since calibration error from the representative dataset affects both weights and activations.

**3. Did QAT improve the final int8 model compared with PTQ int8?**
Yes, QAT is expected to recover most of the accuracy gap between PTQ int8 and the FP32 baseline. Because the network is fine-tuned with simulated quantization noise already present during training, the weights adapt to reduced precision instead of being quantized after the fact, which is why QAT int8 usually outperforms PTQ int8 at the same model size.

**4. Why is this dataset a good fit for a DNN-based TinyML workflow?**
UCI HAR is already reduced to 561 hand-engineered numerical features per sample, so there's no spatial or sequential structure left for a CNN or RNN to exploit, a compact feedforward network is sufficient. That keeps the model small from the start, which is exactly the profile TinyML targets: low compute, low memory, and a classification task simple enough to tolerate aggressive quantization without collapsing.

**5. If you were deploying this model on a resource-constrained device, which version would you choose and why?**
QAT int8 if the PTQ int8 accuracy drop is too large for the application, otherwise PTQ int8, it gets the same ~4x size reduction and fast integer only inference microcontrollers need, without the extra training cost of QAT.